# Geographic Bias in Vision-Language Models: Kurdistan as a Case Study

**Research question:** Do vision-language models (VLMs) recognize, name, and geolocate natural landmarks in Iraqi Kurdistan worse than *type-matched* Western landmarks?

**Design**
- **Group A (target):** your KND_v2 cleaned dataset — 39 Kurdistan places.
- **Group B (control):** 39 Western landmarks, each matched to a Kurdistan place by *type* (waterfall↔waterfall, terraced village↔terraced village, ...). Control images are downloaded the same way and used **only for local evaluation** (never redistributed).
- **Models:** (1) CLIP ViT-L/14 and (2) SigLIP — zero-shot closed-set recognition; (3) Qwen2.5-VL (open VLM, runs on your GPU) — open-ended country ID, landmark naming, and 4-way multiple choice; (4) *optional* API models (GPT-4o / Claude) with the same prompts.
- **Metrics:** accuracy per region, Kurdistan−Control gap with cluster-bootstrap 95% CIs (resampling *places*, not images, since images within a place are correlated), hallucination rate (confidently naming a wrong real place), per-type breakdown, and landmark prominence (Wikipedia pageviews) as a covariate.

**Outputs:** `results/` with raw responses (JSONL, resumable), `summary.csv`, publication figures (PNG), and a LaTeX results table.

**Requirements:** NVIDIA GPU (8 GB VRAM is enough with 4-bit; 3B model fits comfortably), the cleaned dataset folder `KND_v2/` next to this notebook, internet.

In [ ]:
%pip install -q ddgs requests pillow imagehash pandas matplotlib tqdm rapidfuzz statsmodels scipy
%pip install -q transformers accelerate bitsandbytes qwen-vl-utils sentencepiece protobuf
# If torch is not a CUDA build:  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import os
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
for _v in ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE'):
    os.environ.pop(_v, None)
import io, re, json, time, random, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import requests
import pandas as pd
import numpy as np
from PIL import Image, ImageFile
from tqdm.auto import tqdm
import torch

ImageFile.LOAD_TRUNCATED_IMAGES = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU only (will be slow)')

KND_ROOT = Path('KND_v2')            # your cleaned Kurdistan dataset
CTRL_ROOT = Path('CONTROL_set')       # matched Western controls (built below)
RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
assert KND_ROOT.exists(), 'Put the extracted KND_v2 folder next to this notebook!'

IMAGES_PER_PLACE_CLIP = 20   # cap per place for CLIP tasks
IMAGES_PER_PLACE_VLM  = 8    # cap per place for generative VLM tasks (runtime!)
SEED = 42
random.seed(SEED); np.random.seed(SEED)

## 1. Study configuration: matched pairs

Each Kurdistan place is paired with a Western landmark of the same *type category*. Controls were chosen to be well-documented but not world-iconic (no Niagara/Eiffel tier) to keep the comparison fair; the Wikipedia-pageview covariate (section 3) quantifies remaining prominence differences. Edit freely — `aliases` are the alternative spellings accepted as correct answers.

In [ ]:
IRAQ = ['iraq', 'kurdistan']

def K(name, cat, aliases=None, wiki=None):
    return dict(name=name, cat=cat, country=IRAQ, aliases=aliases or [], region='kurdistan', wiki=wiki or name)
def C(name, cat, country, aliases=None, wiki=None, q=None):
    return dict(name=name, cat=cat, country=[country.lower()], aliases=aliases or [], region='control', wiki=wiki or name, q=q or name)

PAIRS = [
 (K('Shanidar Cave','cave',['shanidar']), C('Postojna Cave','cave','slovenia',['postojna'],'Postojna Cave')),
 (K('Korek Mountain','mountain_resort',['korek','kurek']), C('Zugspitze','mountain_resort','germany')),
 (K('Sami Abdulrahman Park','park',['sami abdul rahman']), C('Englischer Garten','park','germany',['english garden'],'Englischer Garten','Englischer Garten Munich park')),
 (K('Rawanduz Canyon','canyon',['rawanduz','rowanduz','rwanduz']), C('Verdon Gorge','canyon','france',['gorges du verdon'])),
 (K('Barzan Area','valley',['barzan']), C('Lauterbrunnen Valley','valley','switzerland',['lauterbrunnen'])),
 (K('Soran Area','mountain',['soran']), C('Berchtesgaden','mountain','germany')),
 (K('Hendrin Mountain','mountain',['hendrin','handren']), C('Ben Nevis','mountain','scotland',[],'Ben Nevis','Ben Nevis Scotland mountain')),
 (K('Dukan Lake','lake',['dukan','dokan']), C('Lake Bled','lake','slovenia',['bled'])),
 (K('Azmar Mountain','city_mountain',['azmar','azmer']), C('Arthur\'s Seat','city_mountain','scotland',['arthurs seat'],'Arthur\'s Seat','Arthurs Seat Edinburgh')),
 (K('Ahmad Awa Waterfall','waterfall',['ahmadawa','ahmed awa']), C('Krimml Waterfalls','waterfall','austria',['krimml'])),
 (K('Hawraman Region','terraced_village',['hawraman','hewraman','avroman','uraman']), C('Cinque Terre','terraced_village','italy')),
 (K('Qaradagh Area','hills',['qaradagh','qara dagh']), C('Peak District','hills','england',[],'Peak District','Peak District England hills')),
 (K('Sirwan River Area','river',['sirwan','diyala']), C('Soca River','river','slovenia',['soča'],'Soča','Soca river Slovenia')),
 (K('Amedi','plateau_town',['amadiya','amadiyah','amedy']), C('Ronda','plateau_town','spain',[],'Ronda','Ronda Spain clifftop town')),
 (K('Lalish','sacred_valley',['lalish temple']), C('Meteora','sacred_valley','greece')),
 (K('Akre','mountain_town',['aqrah','akre town']), C('Gruyeres','mountain_town','switzerland',['gruyères'],'Gruyères')),
 (K('Gali Duhok','gorge',['duhok valley','dohuk']), C('Cheddar Gorge','gorge','england')),
 (K('Zakho','historic_bridge_river',['delal bridge','dalal']), C('Mostar','historic_bridge_river','bosnia',['stari most'],'Stari Most','Mostar old bridge')),
 (K('Shaqlawa','mountain_town',['shaqlawa','shaqlava']), C('Zakopane','mountain_town','poland')),
 (K('Darbandikhan Lake','lake',['darbandikhan']), C('Lake Annecy','lake','france',['lac d\'annecy'],'Lake Annecy')),
 (K('Darband Lake (Ranya)','lake',['ranya','darband ranya','darbandi ranya']), C('Lake Wanaka','lake','new zealand',['wanaka'])),
 (K('Gali Ali Bag Waterfall','waterfall',['geli ali beg','gali ali beg','ali beg']), C('Gullfoss','waterfall','iceland')),
 (K('Halgurd Mountain','high_peak',['halgurd','halgord']), C('Mount Olympus','high_peak','greece',['olympus'],'Mount Olympus','Mount Olympus Greece')),
 (K('Przhe Mountain','high_peak',['przhe','pirza']), C('Triglav','high_peak','slovenia')),
 (K('Bekhal Waterfall','waterfall',['bekhal','bexal']), C('Marmore Falls','waterfall','italy',['cascata delle marmore'],'Cascata delle Marmore')),
 (K('Sulav Resort','spring_resort',['sulav','sulaf']), C('Bad Gastein','spring_resort','austria')),
 (K('Zawita Valley','pine_forest',['zawita','zawite']), C('Black Forest','pine_forest','germany',['schwarzwald'],'Black Forest','Black Forest Germany')),
 (K('Gara Mountain','mountain',['gara','matin']), C('Mont Ventoux','mountain','france')),
 (K('Safeen Mountain','ridge',['safeen','safin']), C('Mount Pilatus','ridge','switzerland',['pilatus'])),
 (K('Piramagrun Mountain','rocky_peak',['piramagrun','pira magrun']), C('Tre Cime di Lavaredo','rocky_peak','italy',['tre cime','drei zinnen'])),
 (K('Goizha Mountain','city_mountain',['goizha','goyzha']), C('Mount Royal','city_mountain','canada',['mont royal'],'Mount Royal','Mount Royal Montreal')),
 (K('Penjwen Area','highlands',['penjwen','penjween']), C('Lake District','highlands','england',[],'Lake District','Lake District England')),
 (K('Mergapan Valley','flower_valley',['mergapan','merga pan']), C('Alpe di Siusi','flower_valley','italy',['seiser alm'])),
 (K('Dore Canyon','canyon',['dore','dora canyon']), C('Samaria Gorge','canyon','greece',['samaria'])),
 (K('Great Zab River','river',['great zab','greater zab','zab']), C('Tara River','river','montenegro',['tara canyon'],'Tara (river)','Tara river canyon Montenegro')),
 (K('Khalifan Area','mountain_valley',['khalifan']), C('Zillertal','mountain_valley','austria',['ziller valley'])),
 (K('Chami Rezan Valley','cave_valley',['chami rezan','chami razan']), C('Vezere Valley','cave_valley','france',['vézère'],'Vézère','Vezere valley Dordogne caves')),
 (K('Sartka Mountain','paragliding',['sartka']), C('Oludeniz','paragliding','turkey',['ölüdeniz','babadag'],'Ölüdeniz','Oludeniz Babadag paragliding')),
 (K('Smaquli Valley','valley',['smaquli','smaquli resan']), C('Glen Coe','valley','scotland',['glencoe'])),
]
KURD = [a for a,b in PAIRS]
CTRL = [b for a,b in PAIRS]
print(len(PAIRS), 'matched pairs')

## 2. Build the control image set

Same pipeline as your dataset: DuckDuckGo search → validate → phash dedup. ~20 images per control landmark. Skips already-populated folders, so it's resumable.

In [ ]:
CTRL_ROOT.mkdir(exist_ok=True)

# Skip everything if the control set is already complete
def _count(p):
    d = CTRL_ROOT / p['name'].replace('/', '-')
    return len(list(d.glob('*.jpg'))) if d.exists() else 0

need = [p for p in CTRL if _count(p) < 15]
print(f'{len(CTRL) - len(need)}/{len(CTRL)} control folders already complete')

if not need:
    print('CONTROL_set already exists and is fully populated - skipping all downloads.')
else:
    try:
        from ddgs import DDGS
    except ImportError:
        from duckduckgo_search import DDGS
    import imagehash

    HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/126.0 Safari/537.36'}

    def fetch_one(url, dest, i):
        try:
            r = requests.get(url, headers=HEADERS, timeout=15); r.raise_for_status()
            img = Image.open(io.BytesIO(r.content)); img.load()
            if img.width < 400 or img.height < 300: return
            img = img.convert('RGB')
            if max(img.size) > 1800: img.thumbnail((1800,1800), Image.LANCZOS)
            img.save(dest / f'img_{i:03d}.jpg', 'JPEG', quality=88)
        except Exception:
            pass

    for place in tqdm(need, desc='Control set'):
        dest = CTRL_ROOT / place['name'].replace('/', '-')
        dest.mkdir(exist_ok=True)
        urls = []
        for suffix in ['', ' landscape photo']:
            try:
                with DDGS() as d:
                    for r in d.images(place['q'] + suffix, max_results=30, size='Large') or []:
                        urls.append(r.get('image'))
            except Exception as e:
                print('search retry:', e); time.sleep(5)
            time.sleep(1)
        urls = [u for u in dict.fromkeys(urls) if u][:45]
        with ThreadPoolExecutor(max_workers=10) as ex:
            list(ex.map(lambda t: fetch_one(t[1], dest, t[0]), enumerate(urls)))
        # phash dedup
        hashes = []
        for f in sorted(dest.glob('*.jpg')):
            try:
                h = imagehash.phash(Image.open(f))
                if any(h - h2 <= 6 for h2 in hashes): f.unlink()
                else: hashes.append(h)
            except Exception: f.unlink(missing_ok=True)
        print(place['name'], len(list(dest.glob('*.jpg'))))

## 3. Evaluation manifest + prominence covariate

One row per evaluation image, capped per place, with a fixed random sample (seeded → reproducible). Then Wikipedia pageviews per landmark as a fame covariate — this lets you *report* the prominence difference instead of pretending it doesn't exist.

In [ ]:
def collect(places, root, is_kurd):
    rows = []
    for p in places:
        folder = root / (p['name'] if is_kurd else p['name'].replace('/', '-'))
        imgs = sorted(folder.glob('*.jpg'))
        if not imgs:
            print('WARNING: no images for', p['name']); continue
        sel = random.Random(SEED).sample(imgs, min(IMAGES_PER_PLACE_CLIP, len(imgs)))
        for f in sel:
            rows.append(dict(path=str(f), place=p['name'], region=p['region'], cat=p['cat']))
    return rows

manifest = pd.DataFrame(collect(KURD, KND_ROOT, True) + collect(CTRL, CTRL_ROOT, False))
manifest.to_csv(RESULTS/'manifest.csv', index=False)
print(manifest.groupby('region').size())

# --- Wikipedia pageviews (fame covariate); safe to skip if it errors ---
def pageviews(title):
    t = requests.utils.quote(title.replace(' ', '_'), safe='')
    url = (f'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/'
           f'en.wikipedia/all-access/user/{t}/monthly/2025010100/2026010100')
    try:
        r = requests.get(url, headers={'User-Agent':'geo-bias-research'}, timeout=10)
        return sum(x['views'] for x in r.json().get('items', []))
    except Exception:
        return None

fame = {p['name']: pageviews(p['wiki']) for p in tqdm(KURD + CTRL, desc='pageviews')}
pd.Series(fame, name='annual_pageviews').to_csv(RESULTS/'prominence.csv')
kv = [v for p,v in fame.items() if v and any(p==k['name'] for k in KURD)]
cv = [v for p,v in fame.items() if v and any(p==c['name'] for c in CTRL)]
print(f'median pageviews  kurdistan={np.median(kv):,.0f}  control={np.median(cv):,.0f}')

## 4. Task 1 — CLIP & SigLIP zero-shot recognition (closed-set)

For each region separately: can the model pick the correct place among that region's 39 candidates? If VLM pretraining underrepresents Kurdistan, accuracy should be systematically lower there — with an identical protocol and identical chance level (1/39).

In [ ]:
from transformers import AutoModel, AutoProcessor, CLIPModel, CLIPProcessor

def _feat(x):
    """Handle transformers versions where get_*_features returns an output object."""
    if torch.is_tensor(x):
        return x
    if hasattr(x, 'pooler_output') and x.pooler_output is not None:
        return x.pooler_output
    return x[0] if torch.is_tensor(x[0]) else x[1]

def zero_shot(model_id, kind):
    if kind == 'clip':
        model = CLIPModel.from_pretrained(model_id, use_safetensors=True, torch_dtype=torch.float16).to(DEVICE).eval()
        proc = CLIPProcessor.from_pretrained(model_id)
    else:  # siglip
        model = AutoModel.from_pretrained(model_id, use_safetensors=True, torch_dtype=torch.float16).to(DEVICE).eval()
        proc = AutoProcessor.from_pretrained(model_id)
    out = []
    for region, places in [('kurdistan', KURD), ('control', CTRL)]:
        names = [p['name'] for p in places]
        prompts = [f'a photo of {n}' for n in names]
        with torch.no_grad():
            ti = proc(text=prompts, return_tensors='pt', padding=True, truncation=True).to(DEVICE)
            try:
                tfeat = _feat(model.get_text_features(**ti))
            except TypeError:
                # some SigLIP processor versions return extra keys the text tower rejects
                ti = {k: v for k, v in ti.items() if k in ('input_ids', 'attention_mask')}
                tfeat = _feat(model.get_text_features(**ti))
            tfeat = tfeat / tfeat.norm(dim=-1, keepdim=True)
        sub = manifest[manifest.region == region]
        for i in tqdm(range(0, len(sub), 32), desc=f'{model_id} {region}'):
            batch = sub.iloc[i:i+32]
            imgs = [Image.open(p).convert('RGB') for p in batch.path]
            with torch.no_grad():
                ii = proc(images=imgs, return_tensors='pt').to(DEVICE)
                if_ = _feat(model.get_image_features(
                    **{k: v.half() if v.dtype == torch.float32 else v for k, v in ii.items()}))
                if_ = if_ / if_.norm(dim=-1, keepdim=True)
                sims = (if_ @ tfeat.T).float().cpu().numpy()
            for (_, row), s in zip(batch.iterrows(), sims):
                order = np.argsort(-s)
                out.append(dict(model=model_id, task='closed_set', path=row.path, place=row.place,
                                region=region, cat=row['cat'],
                                pred=names[order[0]],
                                correct=int(names[order[0]] == row.place),
                                top5=int(row.place in [names[j] for j in order[:5]])))
    del model; torch.cuda.empty_cache()
    return pd.DataFrame(out)

if (RESULTS/'closed_set.csv').exists():
    clip_all = pd.read_csv(RESULTS/'closed_set.csv')
    print('closed_set.csv already exists - loaded from disk, skipping model runs.')
else:
    clip_res = zero_shot('openai/clip-vit-large-patch14', 'clip')
    siglip_res = zero_shot('google/siglip-so400m-patch14-384', 'siglip')
    clip_all = pd.concat([clip_res, siglip_res])
    clip_all.to_csv(RESULTS/'closed_set.csv', index=False)
print(clip_all.groupby(['model','region'])[['correct','top5']].mean().round(3))

## 5. Tasks 2–4 — open VLM (Qwen2.5-VL) on your GPU

Three prompts per image: **country ID**, **open landmark naming**, **4-way MCQ**. Runs in 4-bit; responses stream to a JSONL so you can stop/resume anytime. With 8 images/place this is ~1,850 generations ≈ 1.5–3 h on a laptop GPU.

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# --- pick the largest model your machine can actually load ---
# Loading 7B in 4-bit briefly needs ~20 GB of COMMITTED memory (RAM + pagefile) on Windows;
# if that's not available, Windows kills the kernel silently. We check first instead of crashing.
try:
    import psutil
    avail_gb = (psutil.virtual_memory().available + psutil.swap_memory().free) / 1e9
except ImportError:
    avail_gb = None
    print('tip: pip install psutil for automatic model selection')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0

if (avail_gb is None or avail_gb >= 22) and vram_gb >= 9:
    VLM_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'
else:
    VLM_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
    print(f'Available RAM+swap = {avail_gb:.1f} GB, VRAM = {vram_gb:.1f} GB -> using the 3B model.')
    print('To use 7B: enlarge the Windows pagefile to 32 GB (see instructions) and restart.')
print('Loading:', VLM_ID)

# Make sure ALL repo files are cached first. snapshot_download resumes through
# connection drops and rate limits, unlike the loaders' one-shot HEAD checks.
from huggingface_hub import snapshot_download
for attempt in range(5):
    try:
        snapshot_download(VLM_ID, max_workers=4)
        print('all model files cached')
        break
    except Exception as e:
        print(f'download interrupted ({e}); resuming in 10s [{attempt+1}/5]')
        time.sleep(10)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
# local_files_only=True: load purely from cache - no network calls, nothing left to fail
vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(VLM_ID, quantization_config=bnb,
                                                         device_map='auto', low_cpu_mem_usage=True,
                                                         local_files_only=True)
vproc = AutoProcessor.from_pretrained(VLM_ID, min_pixels=256*28*28, max_pixels=768*28*28,
                                      local_files_only=True)

def ask(img_path, question, max_new=60):
    msgs = [{'role':'user','content':[{'type':'image','image':str(img_path)},{'type':'text','text':question}]}]
    text = vproc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(msgs)
    inputs = vproc(text=[text], images=imgs, videos=vids, return_tensors='pt').to(vlm.device)
    with torch.no_grad():
        ids = vlm.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    ans = vproc.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    return ans.strip()

Q_COUNTRY = 'In which country was this photo taken? Answer with the country name only.'
Q_NAME    = 'What is the name of this specific place or landmark? If you are not sure, say "unsure". Answer with the name only.'

def mcq_options(place, region_places):
    others = [p['name'] for p in region_places if p['name'] != place]
    opts = random.Random(hash(place) & 0xffff).sample(others, 3) + [place]
    random.Random(hash(place) >> 4 & 0xffff).shuffle(opts)
    return opts

vlm_manifest = manifest.groupby('place').head(IMAGES_PER_PLACE_VLM).reset_index(drop=True)
OUTJL = RESULTS / 'vlm_responses.jsonl'
done = set()
if OUTJL.exists():
    for l in OUTJL.read_text(encoding='utf-8').splitlines():
        try: done.add(json.loads(l)['path'])
        except Exception: pass

with OUTJL.open('a', encoding='utf-8', buffering=1) as out:
    for _, row in tqdm(vlm_manifest.iterrows(), total=len(vlm_manifest), desc='VLM'):
        if row.path in done: continue
        places = KURD if row.region=='kurdistan' else CTRL
        opts = mcq_options(row.place, places)
        letters = ['A','B','C','D']
        q_mcq = ('Which place is shown in this photo?\n' +
                 '\n'.join(f'{l}. {o}' for l,o in zip(letters,opts)) +
                 '\nAnswer with the letter only.')
        rec = dict(path=row.path, place=row.place, region=row.region, cat=row['cat'],
                   model=VLM_ID,
                   country_ans=ask(row.path, Q_COUNTRY, 15),
                   name_ans=ask(row.path, Q_NAME, 40),
                   mcq_opts=opts, mcq_gold=letters[opts.index(row.place)],
                   mcq_ans=ask(row.path, q_mcq, 5))
        out.write(json.dumps(rec, ensure_ascii=False) + '\n')
print('VLM responses saved to', OUTJL)

## 5b. Shared runner + country-hint experiment

`run_generative()` lets any model (open or API) go through the identical three tasks.  
The **hint experiment** re-asks the naming question with the country revealed ("This photo was taken in Iraqi Kurdistan / Slovenia / ..."). If accuracy jumps, the landmark knowledge exists but isn't retrieved from pixels alone; if it stays near zero, the knowledge is absent from the model. Either way it's a finding.

In [ ]:
def run_generative(ask_fn, model_tag, out_path, per_place=IMAGES_PER_PLACE_VLM):
    """Run country/naming/MCQ tasks for any ask(img_path, question)->str function. Resumable."""
    sub = manifest.groupby('place').head(per_place).reset_index(drop=True)
    out_path = Path(out_path)
    done = set()
    if out_path.exists():
        for l in out_path.read_text(encoding='utf-8').splitlines():
            try: done.add(json.loads(l)['path'])
            except Exception: pass
    letters = ['A','B','C','D']
    with out_path.open('a', encoding='utf-8', buffering=1) as out:
        for _, row in tqdm(sub.iterrows(), total=len(sub), desc=model_tag):
            if row['path'] in done: continue
            places = KURD if row['region']=='kurdistan' else CTRL
            opts = mcq_options(row['place'], places)
            q_mcq = ('Which place is shown in this photo?\n' +
                     '\n'.join(f'{l}. {o}' for l,o in zip(letters,opts)) +
                     '\nAnswer with the letter only.')
            try:
                rec = dict(path=row['path'], place=row['place'], region=row['region'], cat=row['cat'],
                           model=model_tag,
                           country_ans=ask_fn(row['path'], Q_COUNTRY),
                           name_ans=ask_fn(row['path'], Q_NAME),
                           mcq_opts=opts, mcq_gold=letters[opts.index(row['place'])],
                           mcq_ans=ask_fn(row['path'], q_mcq))
                out.write(json.dumps(rec, ensure_ascii=False) + '\n')
            except Exception as e:
                print('error:', e); time.sleep(3)

# ---------- Country-hint experiment (runs on the Qwen model already in memory) ----------
HINT_COUNTRY = {p['name']: ('Iraqi Kurdistan' if p['region']=='kurdistan' else p['country'][0].title())
                for p in KURD + CTRL}
HINTJL = RESULTS / 'hint_responses.jsonl'
done_h = set()
if HINTJL.exists():
    done_h = {json.loads(l)['path'] for l in HINTJL.read_text(encoding='utf-8').splitlines()}
sub = manifest.groupby('place').head(IMAGES_PER_PLACE_VLM).reset_index(drop=True)
with HINTJL.open('a', encoding='utf-8', buffering=1) as out:
    for _, row in tqdm(sub.iterrows(), total=len(sub), desc='hint'):
        if row['path'] in done_h: continue
        q = (f"This photo was taken in {HINT_COUNTRY[row['place']]}. "
             'What is the name of this specific place or landmark? '
             'If you are not sure, say "unsure". Answer with the name only.')
        rec = dict(path=row['path'], place=row['place'], region=row['region'], cat=row['cat'],
                   model=VLM_ID + '+hint', name_ans=ask(row['path'], q, 40))
        out.write(json.dumps(rec, ensure_ascii=False) + '\n')
print('hint responses saved')

## 6. Optional — API frontier models (GPT-4o / Claude)

Same prompts against commercial models makes the paper much stronger ("the bias persists at the frontier"). Set a key to enable; ~600 images × 3 questions ≈ a few dollars. Skipped automatically if no key.

In [ ]:
import base64
OPENAI_KEY = os.environ.get('OPENAI_API_KEY', '')   # or paste here

def b64(p):
    im = Image.open(p).convert('RGB'); im.thumbnail((1024,1024))
    buf = io.BytesIO(); im.save(buf,'JPEG',quality=85)
    return base64.b64encode(buf.getvalue()).decode()

def ask_gpt(p, q):
    r = requests.post('https://api.openai.com/v1/chat/completions',
        headers={'Authorization': f'Bearer {OPENAI_KEY}'},
        json={'model':'gpt-4o','max_tokens':60,'messages':[{'role':'user','content':[
            {'type':'text','text':q},
            {'type':'image_url','image_url':{'url':f'data:image/jpeg;base64,{b64(p)}'}}]}]},
        timeout=60)
    return r.json()['choices'][0]['message']['content'].strip()

if OPENAI_KEY:
    OUTJL2 = RESULTS / 'gpt4o_responses.jsonl'
    done2 = set()
    if OUTJL2.exists():
        done2 = {json.loads(l)['path'] for l in OUTJL2.read_text(encoding='utf-8').splitlines()}
    sub = vlm_manifest.groupby('place').head(4).reset_index(drop=True)  # 4/place to control cost
    with OUTJL2.open('a', encoding='utf-8', buffering=1) as out:
        for _, row in tqdm(sub.iterrows(), total=len(sub), desc='GPT-4o'):
            if row.path in done2: continue
            places = KURD if row.region=='kurdistan' else CTRL
            opts = mcq_options(row.place, places); letters=['A','B','C','D']
            q_mcq = ('Which place is shown in this photo?\n' + '\n'.join(f'{l}. {o}' for l,o in zip(letters,opts)) + '\nAnswer with the letter only.')
            try:
                rec = dict(path=row.path, place=row.place, region=row.region, cat=row['cat'], model='gpt-4o',
                           country_ans=ask_gpt(row.path, Q_COUNTRY), name_ans=ask_gpt(row.path, Q_NAME),
                           mcq_opts=opts, mcq_gold=letters[opts.index(row.place)], mcq_ans=ask_gpt(row.path, q_mcq))
                out.write(json.dumps(rec, ensure_ascii=False)+'\n')
            except Exception as e:
                print('api error', e); time.sleep(5)
else:
    print('No OPENAI_API_KEY set - skipping frontier model eval (the paper works without it).')

## 6b. Optional — Gemini API

Second frontier model. Get a free API key at https://aistudio.google.com/apikey (the free tier covers this run). Skipped automatically if no key.

In [ ]:
GEMINI_KEY = os.environ.get('GEMINI_API_KEY', '')   # or paste here

def ask_gemini(p, q):
    url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={GEMINI_KEY}'
    body = {'contents': [{'parts': [{'text': q},
            {'inline_data': {'mime_type': 'image/jpeg', 'data': b64(p)}}]}],
            'generationConfig': {'maxOutputTokens': 60, 'temperature': 0}}
    r = requests.post(url, json=body, timeout=60)
    return r.json()['candidates'][0]['content']['parts'][0]['text'].strip()

if GEMINI_KEY:
    run_generative(ask_gemini, 'gemini-2.5-flash', RESULTS / 'gemini_responses.jsonl', per_place=4)
else:
    print('No GEMINI_API_KEY set - skipping.')

## 7. Scoring & statistics

- **Country / naming** scored with alias-aware fuzzy matching (rapidfuzz).
- **Hallucination** = model names a *different specific place* instead of saying "unsure".
- **Cluster bootstrap** (resample places, 2,000 iters) for the Kurdistan−Control gap and its 95% CI, plus a two-proportion z-test. Resampling at place level is the statistically correct choice because images of the same place are not independent.

In [ ]:
from rapidfuzz import fuzz
from scipy import stats as sps  # if missing: %pip install scipy

ALIAS = {p['name']: [p['name']] + p['aliases'] for p in KURD + CTRL}
COUNTRY = {p['name']: p['country'] for p in KURD + CTRL}
ALL_NAMES = {n.lower() for p in KURD+CTRL for n in [p['name']]+p['aliases']}

def norm(s): return re.sub(r'[^a-z0-9 ]',' ', str(s).lower()).strip()

def score_rows(jl_path):
    rows = [json.loads(l) for l in Path(jl_path).read_text(encoding='utf-8').splitlines()]
    for r in rows:
        ca = norm(r['country_ans'])
        r['country_ok'] = int(any(c in ca for c in COUNTRY[r['place']]))
        na = norm(r['name_ans'])
        gold = [norm(a) for a in ALIAS[r['place']]]
        r['name_ok'] = int(any(g in na or fuzz.partial_ratio(g, na) >= 88 for g in gold))
        r['abstain'] = int('unsure' in na or 'not sure' in na or "don't know" in na or na=='')
        r['halluc'] = int(not r['name_ok'] and not r['abstain'] and len(na) > 3)
        r['mcq_ok'] = int(norm(r['mcq_ans'])[:1] == r['mcq_gold'].lower())
    return pd.DataFrame(rows)

# NOTE: put ALL model response files in results/ before running this, e.g.
#   vlm_responses.jsonl (3B), vlm7b_responses.jsonl (7B), gemini_responses.jsonl, gpt4o_responses.jsonl
frames = [score_rows(jl) for jl in sorted(RESULTS.glob('*_responses.jsonl')) if 'hint' not in jl.name]
gen = pd.concat(frames, ignore_index=True)
_models = list(gen.model.unique())
# the model used for the hint comparison / regression figures must match the hint file's model (the 3B)
MAIN_VLM = next((m for m in _models if '3B' in m), _models[0])
print('models scored:', _models)
print('MAIN_VLM for detailed analyses:', MAIN_VLM)
gen.to_csv(RESULTS/'generative_scored.csv', index=False)

def cluster_boot(df, col, iters=2000):
    """95% CI for the control-minus-Kurdistan gap, resampling places (clusters)."""
    gaps = []
    kp = df[df.region=='kurdistan'].place.unique(); cp = df[df.region=='control'].place.unique()
    rng = np.random.default_rng(SEED)
    for _ in range(iters):
        k_sample = rng.choice(kp, len(kp), replace=True)
        c_sample = rng.choice(cp, len(cp), replace=True)
        kacc = np.mean([df[(df.place==p)][col].mean() for p in k_sample])
        cacc = np.mean([df[(df.place==p)][col].mean() for p in c_sample])
        gaps.append(cacc - kacc)
    return np.percentile(gaps, [2.5, 50, 97.5])

summary = []
for (model,), g in list(gen.groupby(['model'])) :
    for col, label in [('country_ok','Country ID'),('name_ok','Landmark naming'),('mcq_ok','4-way MCQ'),('halluc','Hallucination rate')]:
        kacc = g[g.region=='kurdistan'][col].mean(); cacc = g[g.region=='control'][col].mean()
        lo, med, hi = cluster_boot(g, col)
        n1 = (g.region=='kurdistan').sum(); n2 = (g.region=='control').sum()
        k_hits = int(g[g.region=='kurdistan'][col].sum()); c_hits = int(g[g.region=='control'][col].sum())
        table = np.array([[c_hits, n2 - c_hits], [k_hits, n1 - k_hits]])
        chi2, pval = sps.chi2_contingency(table)[:2]
        summary.append(dict(model=model, task=label, kurdistan=round(kacc,3), control=round(cacc,3),
                            gap=round(cacc-kacc,3), ci_low=round(lo,3), ci_high=round(hi,3), p=f'{pval:.1e}'))
for (model,), g in clip_all.groupby(['model']):
    for col, label in [('correct','Closed-set top-1'),('top5','Closed-set top-5')]:
        kacc = g[g.region=='kurdistan'][col].mean(); cacc = g[g.region=='control'][col].mean()
        lo, med, hi = cluster_boot(g, col)
        summary.append(dict(model=model, task=label, kurdistan=round(kacc,3), control=round(cacc,3),
                            gap=round(cacc-kacc,3), ci_low=round(lo,3), ci_high=round(hi,3), p=''))
summary = pd.DataFrame(summary)
summary.to_csv(RESULTS/'summary.csv', index=False)
summary

## 8. Figures & LaTeX table

In [ ]:
import matplotlib.pyplot as plt

# Fig 1: accuracy by region per task (main VLM)
main = gen[gen.model==MAIN_VLM]
tasks = [('country_ok','Country ID'),('name_ok','Naming'),('mcq_ok','MCQ')]
fig, ax = plt.subplots(figsize=(7,4))
x = np.arange(len(tasks)); w = 0.35
kv = [main[main.region=='kurdistan'][c].mean() for c,_ in tasks]
cv = [main[main.region=='control'][c].mean() for c,_ in tasks]
ax.bar(x-w/2, cv, w, label='Matched Western controls', color='#4878CF')
ax.bar(x+w/2, kv, w, label='Kurdistan', color='#D65F5F')
ax.set_xticks(x); ax.set_xticklabels([t for _,t in tasks]); ax.set_ylabel('Accuracy')
ax.set_title(f'Recognition gap ({MAIN_VLM.split("/")[-1]})'); ax.legend(); ax.set_ylim(0,1)
for i,(a,b) in enumerate(zip(cv,kv)):
    ax.text(i, max(a,b)+0.03, f'Δ={a-b:+.2f}', ha='center')
plt.tight_layout(); plt.savefig(RESULTS/'fig1_gap.png', dpi=300); plt.show()

# Fig 2: per-place naming accuracy vs Wikipedia prominence
fam = pd.read_csv(RESULTS/'prominence.csv', index_col=0)['annual_pageviews']
pp = main.groupby(['place','region'])['name_ok'].mean().reset_index()
pp['views'] = pp.place.map(fam)
fig, ax = plt.subplots(figsize=(7,4.5))
for reg, col in [('control','#4878CF'),('kurdistan','#D65F5F')]:
    d = pp[(pp.region==reg) & pp.views.notna() & (pp.views>0)]
    ax.scatter(d.views, d.name_ok, c=col, label=reg, alpha=0.8)
ax.set_xscale('log'); ax.set_xlabel('Wikipedia pageviews / year (log)'); ax.set_ylabel('Naming accuracy')
ax.set_title('Prominence vs recognition'); ax.legend()
plt.tight_layout(); plt.savefig(RESULTS/'fig2_prominence.png', dpi=300); plt.show()

# LaTeX table (no jinja2 dependency)
def df_to_latex(df):
    cols = ' & '.join(df.columns) + r' \\'
    rows = '\n'.join(' & '.join(str(v) for v in r) + r' \\' for r in df.values)
    return ('\\begin{table}[t]\\centering\n'
            '\\caption{Vision-language model performance on Kurdistan landmarks vs '
            'type-matched Western controls. Gap = control $-$ Kurdistan; '
            'CI from cluster bootstrap over places.}\n'
            '\\label{tab:main}\n'
            '\\begin{tabular}{ll' + 'r'*(len(df.columns)-2) + '}\n\\hline\n'
            + cols + '\n\\hline\n' + rows + '\n\\hline\n\\end{tabular}\\end{table}\n')

with open(RESULTS/'table1.tex', 'w') as f:
    f.write(df_to_latex(summary))
print('Saved: fig1_gap.png, fig2_prominence.png, table1.tex, summary.csv')

## 8b. Closing the reviewer holes

Three analyses that preempt the main objections:
1. **Prominence-controlled regression** — is the gap explained by fame? Logistic regression of naming accuracy on log-pageviews + region with place-clustered SEs. A significant region coefficient = bias beyond prominence.
2. **Country-confusion analysis** — where does the model *think* Kurdistan is? (Fig 3)
3. **Hint effect** — does revealing the country unlock the landmark name? (Fig 4)

In [ ]:
import statsmodels.api as sm
fam = pd.read_csv(RESULTS/'prominence.csv', index_col=0)['annual_pageviews']
main = gen[gen.model == MAIN_VLM].copy()
main['views'] = main.place.map(fam)
reg = main[main.views.notna() & (main.views > 0)].copy()
reg['log_views'] = np.log10(reg.views)
reg['is_kurd'] = (reg.region == 'kurdistan').astype(int)
X = sm.add_constant(reg[['log_views', 'is_kurd']])
fit = sm.Logit(reg['name_ok'], X).fit(disp=0, cov_type='cluster', cov_kwds={'groups': reg['place']})
print(fit.summary2())
or_kurd = np.exp(fit.params['is_kurd'])
print(f"\nOdds ratio for Kurdistan (controlling for prominence): {or_kurd:.3f}")
print("If p(is_kurd) < 0.05, the gap is NOT explained by Wikipedia prominence -> headline claim holds.")

# Fig 2b: fitted probability curves over prominence
import matplotlib.pyplot as plt
xs = np.linspace(reg.log_views.min(), reg.log_views.max(), 100)
fig, ax = plt.subplots(figsize=(7, 4.5))
for kurd, col, lab in [(0, '#4878CF', 'control'), (1, '#D65F5F', 'kurdistan')]:
    p = 1/(1+np.exp(-(fit.params['const'] + fit.params['log_views']*xs + fit.params['is_kurd']*kurd)))
    ax.plot(10**xs, p, c=col, lw=2, label=f'{lab} (fitted)')
    d = reg[reg.is_kurd == kurd].groupby('place').agg(v=('views','first'), a=('name_ok','mean'))
    ax.scatter(d.v, d.a, c=col, alpha=0.6)
ax.set_xscale('log'); ax.set_xlabel('Wikipedia pageviews / year'); ax.set_ylabel('Naming accuracy')
ax.set_title('Gap persists at equal prominence'); ax.legend()
plt.tight_layout(); plt.savefig(RESULTS/'fig2b_regression.png', dpi=300); plt.show()

In [ ]:
# Fig 3: which countries does the model assign to Kurdistan images?
def norm_c(s): return re.sub(r'[^a-z ]', '', str(s).lower()).strip()
kc = main[main.region == 'kurdistan'].country_ans.map(norm_c)
counts = kc.value_counts().head(10)
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2E8B57' if c in ('iraq','kurdistan','iraqi kurdistan') else '#B0B0B0' for c in counts.index]
ax.barh(counts.index[::-1], counts.values[::-1], color=colors[::-1])
ax.set_xlabel('Number of images'); ax.set_title('Countries assigned to Kurdistan landmarks (green = correct)')
plt.tight_layout(); plt.savefig(RESULTS/'fig3_confusion.png', dpi=300); plt.show()
correct_share = kc.isin(['iraq','kurdistan','iraqi kurdistan']).mean()
print(f'Correct-country share: {correct_share:.1%}; ' 
      f"Turkey: {(kc=='turkey').mean():.1%}; Iran: {(kc=='iran').mean():.1%}")

In [ ]:
# Fig 4: does revealing the country unlock landmark names?
hint = [json.loads(l) for l in (RESULTS/'hint_responses.jsonl').read_text(encoding='utf-8').splitlines()]
hint = pd.DataFrame(hint)
def name_ok_row(r):
    na = norm(r['name_ans']); gold = [norm(a) for a in ALIAS[r['place']]]
    return int(any(g in na or fuzz.partial_ratio(g, na) >= 88 for g in gold))
hint['name_ok'] = hint.apply(name_ok_row, axis=1)
cmp = pd.DataFrame({
    'no hint': main.groupby('region')['name_ok'].mean(),
    'with country hint': hint.groupby('region')['name_ok'].mean()})
print(cmp.round(3))
ax = cmp.plot.bar(figsize=(6,4), color=['#888888', '#2E8B57'], rot=0)
ax.set_ylabel('Naming accuracy'); ax.set_title('Effect of revealing the country')
plt.tight_layout(); plt.savefig(RESULTS/'fig4_hint.png', dpi=300); plt.show()
gain_k = cmp.loc['kurdistan','with country hint'] - cmp.loc['kurdistan','no hint']
print(f"Kurdistan gain from hint: {gain_k:+.3f} -> " +
      ("knowledge exists but is not retrieved from pixels alone" if gain_k > 0.1 else
       "knowledge is largely absent from the model"))

## 8c. Model-scale analysis: confabulation vs abstention

The 7B and 3B fail differently on Kurdistan: the larger model abstains ("unsure"), the smaller one confidently names the wrong place. This cell quantifies that scale-dependent failure mode (Fig 5) - only runs if both models' response files are in `results/`.

In [ ]:
qwens = [m for m in gen.model.unique() if 'Qwen' in m]
if len(qwens) >= 2:
    sc = (gen[gen.model.isin(qwens)]
          .groupby(['model','region'])[['name_ok','abstain','halluc']].mean().round(3))
    print(sc)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for ax, metric, title in zip(axes, ['abstain','halluc'], ['Abstention ("unsure")','Hallucination (wrong place named)']):
        w = 0.35
        for j, m in enumerate(sorted(qwens)):
            d = gen[gen.model==m]
            vals = [d[d.region=='control'][metric].mean(), d[d.region=='kurdistan'][metric].mean()]
            ax.bar(np.arange(2)+ (j-0.5)*w, vals, w, label=m.split('/')[-1].replace('-Instruct',''))
        ax.set_xticks([0,1]); ax.set_xticklabels(['Control','Kurdistan'])
        ax.set_title(title); ax.set_ylim(0,1)
    axes[0].set_ylabel('Rate'); axes[0].legend()
    plt.tight_layout(); plt.savefig(RESULTS/'fig5_scale.png', dpi=300); plt.show()
    print("Reading: larger model abstains on Kurdistan; smaller model confabulates. "
          "Scale changes the failure mode, not the knowledge gap.")
else:
    print('Only one Qwen model found - add vlm7b_responses.jsonl to results/ and re-run scoring.')

## 8d. Additional publication figures (all 300 dpi)

Fig 6: all-models overview; Fig 7: matched-pair dumbbells (CLIP); Fig 8: gap by place type; Fig 9: per-place profile of Kurdistan; Fig 10: failure-mode composition. All saved as PNG at 300 dpi in `results/`.

In [ ]:
# ---------- Fig 6: all models x all tasks overview ----------
plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 300, 'font.size': 10})
KCOL, CCOL = '#D65F5F', '#4878CF'

rows = []
for m, g in gen.groupby('model'):
    for col, lab in [('country_ok','Country ID'), ('name_ok','Naming'), ('mcq_ok','MCQ')]:
        rows.append(dict(model=m.split('/')[-1].replace('-Instruct',''), task=lab,
                         kurdistan=g[g.region=='kurdistan'][col].mean(),
                         control=g[g.region=='control'][col].mean()))
for m, g in clip_all.groupby('model'):
    rows.append(dict(model=m.split('/')[-1], task='Closed-set top-1',
                     kurdistan=g[g.region=='kurdistan']['correct'].mean(),
                     control=g[g.region=='control']['correct'].mean()))
ov = pd.DataFrame(rows)
tasks_order = ['Closed-set top-1','Country ID','Naming','MCQ']
models_order = list(dict.fromkeys(ov.model))
fig, axes = plt.subplots(1, len(tasks_order), figsize=(4*len(tasks_order), 4.2), sharey=True)
for ax, t in zip(axes, tasks_order):
    d = ov[ov.task==t].set_index('model').reindex(models_order).dropna()
    y = np.arange(len(d))
    ax.barh(y+0.2, d.control, 0.38, color=CCOL, label='Control')
    ax.barh(y-0.2, d.kurdistan, 0.38, color=KCOL, label='Kurdistan')
    ax.set_yticks(y); ax.set_yticklabels(d.index); ax.set_xlim(0,1)
    ax.set_title(t); ax.set_xlabel('Accuracy')
axes[0].legend(loc='lower right')
plt.tight_layout(); plt.savefig(RESULTS/'fig6_overview.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
# ---------- Fig 7: matched-pair dumbbells (CLIP top-1 per pair) ----------
clip_l = clip_all[clip_all.model=='openai/clip-vit-large-patch14']
acc = clip_l.groupby(['place','region'])['correct'].mean()
pairs = [(k['name'], c['name']) for k, c in PAIRS]
rows = [(kn, cn, acc.get((kn,'kurdistan'), np.nan), acc.get((cn,'control'), np.nan)) for kn, cn in pairs]
pp = pd.DataFrame(rows, columns=['kurd','ctrl','k_acc','c_acc']).dropna().sort_values('c_acc')
fig, ax = plt.subplots(figsize=(7, 10))
y = np.arange(len(pp))
for i, r in enumerate(pp.itertuples()):
    ax.plot([r.k_acc, r.c_acc], [i, i], color='#BBBBBB', lw=1.6, zorder=1)
ax.scatter(pp.c_acc, y, color=CCOL, s=42, zorder=2, label='Control landmark')
ax.scatter(pp.k_acc, y, color=KCOL, s=42, zorder=2, label='Kurdistan landmark')
ax.set_yticks(y); ax.set_yticklabels([f'{r.kurd}  /  {r.ctrl}' for r in pp.itertuples()], fontsize=7.5)
ax.set_xlabel('CLIP closed-set top-1 accuracy'); ax.set_xlim(-0.03, 1.03)
ax.set_title('Matched pairs: recognition of each Kurdistan landmark vs its Western counterpart')
ax.legend(loc='lower right')
plt.tight_layout(); plt.savefig(RESULTS/'fig7_pairs.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
# ---------- Fig 8: gap by place type ----------
main = gen[gen.model==MAIN_VLM]
TYPE_GROUP = {'waterfall':'Waterfall','lake':'Lake','river':'River','canyon':'Canyon/Gorge','gorge':'Canyon/Gorge',
 'cave':'Cave','cave_valley':'Cave','mountain':'Mountain','high_peak':'Mountain','rocky_peak':'Mountain',
 'ridge':'Mountain','city_mountain':'Mountain','mountain_resort':'Mountain','paragliding':'Mountain',
 'valley':'Valley','mountain_valley':'Valley','flower_valley':'Valley','sacred_valley':'Valley',
 'town':'Town/Village','mountain_town':'Town/Village','plateau_town':'Town/Village','terraced_village':'Town/Village',
 'historic_bridge_river':'Town/Village','park':'Park/Hills','hills':'Park/Hills','highlands':'Park/Hills',
 'pine_forest':'Park/Hills','spring_resort':'Park/Hills'}
main = main.assign(tg=main['cat'].map(lambda c: TYPE_GROUP.get(c, 'Other')))
gt = main.groupby(['tg','region'])['mcq_ok'].mean().unstack()
gt['gap'] = gt['control'] - gt['kurdistan']
gt = gt.sort_values('gap')
fig, ax = plt.subplots(figsize=(7, 4.5))
y = np.arange(len(gt))
ax.barh(y, gt['gap'], color='#8B5A83')
ax.set_yticks(y); ax.set_yticklabels(gt.index)
ax.set_xlabel('MCQ accuracy gap (control - Kurdistan)')
ax.set_title(f'Where the gap is largest, by landmark type ({MAIN_VLM.split("/")[-1]})')
for i, v in enumerate(gt['gap']):
    ax.text(v + 0.01, i, f'{v:+.2f}', va='center', fontsize=8)
plt.tight_layout(); plt.savefig(RESULTS/'fig8_types.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
# ---------- Fig 9: per-place profile for Kurdistan ----------
k = gen[(gen.model==MAIN_VLM) & (gen.region=='kurdistan')]
prof = k.groupby('place')[['country_ok','name_ok','mcq_ok']].mean().sort_values('mcq_ok')
fig, ax = plt.subplots(figsize=(7, 9))
y = np.arange(len(prof))
ax.scatter(prof.mcq_ok, y, color='#2E8B57', s=40, label='MCQ (4-way)')
ax.scatter(prof.country_ok, y, color='#E8A33D', s=40, label='Country ID')
ax.scatter(prof.name_ok, y, color=KCOL, s=40, label='Open naming')
for i in y: ax.axhline(i, color='#EEEEEE', lw=0.8, zorder=0)
ax.set_yticks(y); ax.set_yticklabels(prof.index, fontsize=8)
ax.set_xlim(-0.03, 1.03); ax.set_xlabel('Accuracy')
ax.set_title('Kurdistan places: scaffolded recognition vs open naming')
ax.legend(loc='lower right')
plt.tight_layout(); plt.savefig(RESULTS/'fig9_places.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
# ---------- Fig 10: failure-mode composition (stacked) ----------
comp_rows = []
for m, g in gen.groupby('model'):
    for reg in ['control','kurdistan']:
        d = g[g.region==reg]
        comp_rows.append(dict(model=m.split('/')[-1].replace('-Instruct',''), region=reg,
                              correct=d['name_ok'].mean(), abstain=d['abstain'].mean(),
                              halluc=d['halluc'].mean()))
comp = pd.DataFrame(comp_rows)
comp['other'] = (1 - comp[['correct','abstain','halluc']].sum(axis=1)).clip(lower=0)
labels = [f"{r.model}\n{r.region}" for r in comp.itertuples()]
fig, ax = plt.subplots(figsize=(1.4*len(comp)+2, 4.6))
x = np.arange(len(comp)); bottom = np.zeros(len(comp))
for col, cc, lab in [('correct','#2E8B57','Correct name'), ('abstain','#B8B8B8','Abstains ("unsure")'),
                     ('halluc','#C44E52','Hallucinates wrong place'), ('other','#E8DCC8','Other')]:
    ax.bar(x, comp[col], bottom=bottom, color=cc, label=lab)
    bottom += comp[col].values
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Share of responses'); ax.set_ylim(0, 1)
ax.set_title('Open landmark naming: outcome composition by model and region')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=8)
plt.tight_layout(); plt.savefig(RESULTS/'fig10_failure_modes.png', dpi=300, bbox_inches='tight'); plt.show()
print('Saved fig6-fig10 at 300 dpi in results/')

## 9. Paper skeleton & what to write

**Title idea:** *“Invisible Landscapes: Quantifying Geographic Bias of Vision-Language Models in Iraqi Kurdistan”*

1. **Intro** — VLMs deployed globally, trained on Western-centric web data; geographic fairness matters for tourism tech, education, geolocation services. Gap: no benchmark for Kurdistan / underrepresented Middle East regions.
2. **Related work** — GeoDE, Dollar Street, GeoYFCC (geographic bias in image datasets); WorldBench-style VLM audits; landmark recognition literature.
3. **Method** — the matched-pair design in this notebook; explain place-level cluster bootstrap; prominence covariate; hallucination definition.
4. **Results** — Table 1 + Fig 1 (the gap), Fig 2 (gap not fully explained by prominence — check whether Kurdistan points fall *below* the control trend at equal pageviews: that is your headline finding), qualitative error analysis (what does the model call Amedi? Where does it place Hawraman? — pull examples from `generative_scored.csv`, especially `halluc==1` rows).
5. **Discussion** — harms (misinformation about the region, erasure), why closed-set vs open-ended gaps differ, limitations (web-sourced evaluation images; single-country case study).
6. **Release** — publish the benchmark as *metadata + image URLs + your own photos*, not scraped images.

**Target venues:** ACM FAccT, *Pattern Recognition*, *Information Processing & Management*, NeurIPS Datasets & Benchmarks track, *AI & Society*.

**Strengthen it further (optional):** add a second open VLM (LLaVA-1.6, InternVL) via the same `ask()` harness; add Kurdish/Arabic-language prompts to test cross-lingual bias; have 2 local annotators verify ground truth on a 100-image sample and report agreement.